# 18. Contrastive Decoding：怎样让专家模型减去业余模型的通用偏好？

## 面试回答主线

Contrastive Decoding 同时读取 expert 与 amateur 的下一 token 分布，用 `log p_expert - α log p_amateur` 抑制两个模型都偏爱的空泛高频词。它不是简单选择两模型分歧最大的 token，因为业余模型极低概率的乱码也会被错误奖励。关键修正是在专家模型的 plausibility set 内重排，只允许专家概率不低于其最大概率某个比例的候选参与。面试中我会在同一组中文生成候选上比较 expert argmax、无约束差分和带 mask 的结果，并打印每个 token 的中间分数。该方法需要 expert 与 amateur 共享 tokenizer 和上下文对齐，还会带来额外前向计算与 KV cache 成本。α 与阈值必须在任务验证集上共同调参，不能用一组玩具值宣称普遍更好。

## 1. 真实案例：五类回答的下一 token 候选

每条 prompt 都有四个候选：空泛高频词、领域具体词、次优合理词和一个专家也不认可的怪词。expert/amateur logits 是教学用的可解释快照；候选文本分别对应退款、数据库、RAG、天气和代码审查场景。下面先展示候选与两个模型的原始打分。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示候选 token 分布
import math  # 导入指数和对数函数以手写概率变换
cases = [{"id": "D01", "prompt": "退款申请已审核，下一步应", "tokens": ["可以", "原路退回", "等待", "量子香蕉"], "expert": [3.2, 3.0, 2.2, -2.0], "amateur": [3.8, 1.2, 2.0, -8.0], "desired": "原路退回"}, {"id": "D02", "prompt": "数据库慢查询首先检查", "tokens": ["这个", "执行计划", "日志", "蓝色三角"], "expert": [3.1, 2.9, 2.1, -2.4], "amateur": [3.7, 1.1, 1.9, -8.2], "desired": "执行计划"}, {"id": "D03", "prompt": "RAG 回答需要同时返回", "tokens": ["好的", "来源引用", "文本", "月光端口"], "expert": [3.3, 3.0, 2.0, -2.2], "amateur": [3.9, 1.0, 1.8, -8.5], "desired": "来源引用"}, {"id": "D04", "prompt": "北京暴雨出行应优先", "tokens": ["注意", "查看预警", "带伞", "递归云朵"], "expert": [3.0, 2.8, 2.2, -2.5], "amateur": [3.6, 1.0, 2.0, -8.1], "desired": "查看预警"}, {"id": "D05", "prompt": "代码审查发现密钥后应", "tokens": ["进行", "立即轮换", "删除", "七维饼干"], "expert": [3.2, 3.0, 2.1, -2.3], "amateur": [3.8, 1.1, 1.9, -8.4], "desired": "立即轮换"}]  # 定义五类具有领域语义的候选分布
preview = [{"样本": item["id"], "提示": item["prompt"], "候选": item["tokens"], "expert_logits": item["expert"], "amateur_logits": item["amateur"], "期望": item["desired"]} for item in cases]  # 汇总对比解码需要观察的原始字段
print("Contrastive Decoding 输入预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示五组具体候选与双模型分数

Contrastive Decoding 输入预览：
[{'样本': 'D01',
  '提示': '退款申请已审核，下一步应',
  '候选': ['可以', '原路退回', '等待', '量子香蕉'],
  'expert_logits': [3.2, 3.0, 2.2, -2.0],
  'amateur_logits': [3.8, 1.2, 2.0, -8.0],
  '期望': '原路退回'},
 {'样本': 'D02',
  '提示': '数据库慢查询首先检查',
  '候选': ['这个', '执行计划', '日志', '蓝色三角'],
  'expert_logits': [3.1, 2.9, 2.1, -2.4],
  'amateur_logits': [3.7, 1.1, 1.9, -8.2],
  '期望': '执行计划'},
 {'样本': 'D03',
  '提示': 'RAG 回答需要同时返回',
  '候选': ['好的', '来源引用', '文本', '月光端口'],
  'expert_logits': [3.3, 3.0, 2.0, -2.2],
  'amateur_logits': [3.9, 1.0, 1.8, -8.5],
  '期望': '来源引用'},
 {'样本': 'D04',
  '提示': '北京暴雨出行应优先',
  '候选': ['注意', '查看预警', '带伞', '递归云朵'],
  'expert_logits': [3.0, 2.8, 2.2, -2.5],
  'amateur_logits': [3.6, 1.0, 2.0, -8.1],
  '期望': '查看预警'},
 {'样本': 'D05',
  '提示': '代码审查发现密钥后应',
  '候选': ['进行', '立即轮换', '删除', '七维饼干'],
  'expert_logits': [3.2, 3.0, 2.1, -2.3],
  'amateur_logits': [3.8, 1.1, 1.9, -8.4],
  '期望': '立即轮换'}]


## 2. Baseline（基线）：只做 expert argmax

专家模型已经不错，但高频通用 token 仍略高于具体答案。基线直接取最大 expert logit，因此五个场景都会选到“可以、这个、好的、注意、进行”等空泛开头。后面所有方案仍使用完全相同的 logits，不偷偷修改候选。

In [2]:
baseline_rows = []  # 收集 expert 单模型贪心解码结果
for item in cases:  # 遍历五个真实生成场景
    best_index = max(range(len(item["tokens"])), key=lambda index: item["expert"][index])  # 找到专家模型原始 logit 最大的候选
    choice = item["tokens"][best_index]  # 将最大分索引转换为可读 token
    baseline_rows.append({"样本": item["id"], "提示": item["prompt"], "expert选择": choice, "期望": item["desired"], "具体命中": choice == item["desired"]})  # 保存逐样本基线结果
print("Expert argmax 基线：")  # 标注当前输出属于单模型基线
pprint(baseline_rows, sort_dicts=False)  # 展示专家模型仍偏向哪些通用高频 token

Expert argmax 基线：
[{'样本': 'D01',
  '提示': '退款申请已审核，下一步应',
  'expert选择': '可以',
  '期望': '原路退回',
  '具体命中': False},
 {'样本': 'D02',
  '提示': '数据库慢查询首先检查',
  'expert选择': '这个',
  '期望': '执行计划',
  '具体命中': False},
 {'样本': 'D03',
  '提示': 'RAG 回答需要同时返回',
  'expert选择': '好的',
  '期望': '来源引用',
  '具体命中': False},
 {'样本': 'D04',
  '提示': '北京暴雨出行应优先',
  'expert选择': '注意',
  '期望': '查看预警',
  '具体命中': False},
 {'样本': 'D05',
  '提示': '代码审查发现密钥后应',
  'expert选择': '进行',
  '期望': '立即轮换',
  '具体命中': False}]


## 3. 手写核心算法：稳定 log-softmax 与对比得分

先减最大 logit 再指数化，可以避免数值溢出。对比得分使用两个归一化 log probability，而不是直接减 logits；这样常数平移不会改变结果。下面打印第一条退款样本每个候选的专家概率、业余概率和原始对比得分。

In [3]:
def log_softmax(logits):  # 手写数值稳定的对数 softmax
    maximum = max(logits)  # 找到最大 logit 作为稳定平移常数
    shifted_exp = [math.exp(value - maximum) for value in logits]  # 对平移后的 logits 求指数避免上溢
    log_partition = maximum + math.log(sum(shifted_exp))  # 计算原坐标下的对数归一化常数
    return [value - log_partition for value in logits]  # 返回每个候选的归一化对数概率
alpha = 0.7  # 设置业余模型惩罚强度
def contrastive_scores(item):  # 计算一个场景中所有候选的对比得分
    expert_logp = log_softmax(item["expert"])  # 将专家 logits 转换为对数概率
    amateur_logp = log_softmax(item["amateur"])  # 将业余 logits 转换为对数概率
    scores = [expert_value - alpha * amateur_value for expert_value, amateur_value in zip(expert_logp, amateur_logp)]  # 用专家置信度减去通用模型偏好
    return expert_logp, amateur_logp, scores  # 返回双模型中间量与最终对比分数
expert_logp, amateur_logp, raw_scores = contrastive_scores(cases[0])  # 计算退款场景的候选分数账本
score_ledger = [{"token": token, "expert概率": round(math.exp(expert_value), 4), "amateur概率": round(math.exp(amateur_value), 4), "原始对比分": round(score, 4)} for token, expert_value, amateur_value, score in zip(cases[0]["tokens"], expert_logp, amateur_logp, raw_scores)]  # 对齐四个候选的概率与对比分数
print("退款场景的双模型候选账本：")  # 输出核心算法中间量标题
pprint(score_ledger, sort_dicts=False)  # 展示具体词如何因业余模型低偏好而得到提升

退款场景的双模型候选账本：
[{'token': '可以', 'expert概率': 0.4562, 'amateur概率': 0.8067, '原始对比分': -0.6345},
 {'token': '原路退回', 'expert概率': 0.3735, 'amateur概率': 0.0599, '原始对比分': 0.9855},
 {'token': '等待', 'expert概率': 0.1678, 'amateur概率': 0.1334, '原始对比分': -0.3745},
 {'token': '量子香蕉', 'expert概率': 0.0025, 'amateur概率': 0.0, '原始对比分': 2.4255}]


## 4. Plausibility Mask：只在专家认可的候选内重排

阈值设为专家最大概率的 10%。候选若低于该阈值，即使业余模型概率更低也不能参与 argmax；这保留了专家模型的基本合理性边界。下面手写 mask 并输出每个样本保留下来的候选。

In [4]:
plausibility_ratio = 0.10  # 设置候选至少达到专家最大概率十分之一的门槛
def decode_with_mask(item):  # 在专家可信候选集合内执行对比解码
    expert_logp, amateur_logp, scores = contrastive_scores(item)  # 计算双模型归一化概率与对比分数
    expert_probs = [math.exp(value) for value in expert_logp]  # 将专家对数概率恢复为直观概率
    threshold = max(expert_probs) * plausibility_ratio  # 根据当前上下文的最高专家概率生成动态阈值
    allowed = [probability >= threshold for probability in expert_probs]  # 标记专家仍认为合理的候选集合
    masked_scores = [score if keep else float("-inf") for score, keep in zip(scores, allowed)]  # 把不可信候选排除出最终竞争
    best_index = max(range(len(masked_scores)), key=lambda index: masked_scores[index])  # 在可信集合内选择对比分最高的 token
    kept_tokens = [token for token, keep in zip(item["tokens"], allowed) if keep]  # 收集保留候选帮助解释阈值行为
    return item["tokens"][best_index], kept_tokens, masked_scores  # 返回最终 token、可信集合和带 mask 分数
mask_preview = []  # 收集每个场景的专家可信候选集合
for item in cases:  # 遍历五类生成输入
    choice, kept_tokens, masked_scores = decode_with_mask(item)  # 执行带 plausibility mask 的对比解码
    mask_preview.append({"样本": item["id"], "保留候选": kept_tokens, "最终选择": choice})  # 保存集合与选择供逐样本观察
print("专家可信集合与最终选择：")  # 输出 mask 中间量标题
pprint(mask_preview, sort_dicts=False)  # 展示怪词被过滤而领域具体词胜出

专家可信集合与最终选择：
[{'样本': 'D01', '保留候选': ['可以', '原路退回', '等待'], '最终选择': '原路退回'},
 {'样本': 'D02', '保留候选': ['这个', '执行计划', '日志'], '最终选择': '执行计划'},
 {'样本': 'D03', '保留候选': ['好的', '来源引用', '文本'], '最终选择': '来源引用'},
 {'样本': 'D04', '保留候选': ['注意', '查看预警', '带伞'], '最终选择': '查看预警'},
 {'样本': 'D05', '保留候选': ['进行', '立即轮换', '删除'], '最终选择': '立即轮换'}]


## 5. 结果解读：具体性提升来自惩罚通用偏好

五条样本中，expert argmax 都选空泛 token；带 mask 的对比解码则选中领域具体词。这个结果只能说明当前受控分布的排序机制，不代表所有开放式生成都会改善。逐样本表保留两种选择和专家可信集合，便于识别“具体但仍被专家认可”的边界。

In [5]:
result_rows = []  # 收集同数据上的 expert 基线与对比解码结果
for item, baseline in zip(cases, baseline_rows):  # 对齐原始候选与基线选择
    choice, kept_tokens, masked_scores = decode_with_mask(item)  # 运行带专家可信约束的对比解码
    result_rows.append({"样本": item["id"], "提示": item["prompt"], "expert基线": baseline["expert选择"], "对比解码": choice, "期望": item["desired"], "命中": choice == item["desired"]})  # 保存逐场景业务结果
baseline_hits = sum(row["具体命中"] for row in baseline_rows)  # 统计单专家贪心的具体答案命中数
contrastive_hits = sum(row["命中"] for row in result_rows)  # 统计带 mask 对比解码的具体答案命中数
print("Contrastive Decoding 逐样本对照：")  # 输出结果解读标题
pprint(result_rows, sort_dicts=False)  # 展示五类回答如何从空泛词转向具体词
print(f"具体答案命中：expert 基线 {baseline_hits}/{len(cases)}，带 mask 对比解码 {contrastive_hits}/{len(cases)}")  # 汇总同数据上的可比较收益

Contrastive Decoding 逐样本对照：
[{'样本': 'D01',
  '提示': '退款申请已审核，下一步应',
  'expert基线': '可以',
  '对比解码': '原路退回',
  '期望': '原路退回',
  '命中': True},
 {'样本': 'D02',
  '提示': '数据库慢查询首先检查',
  'expert基线': '这个',
  '对比解码': '执行计划',
  '期望': '执行计划',
  '命中': True},
 {'样本': 'D03',
  '提示': 'RAG 回答需要同时返回',
  'expert基线': '好的',
  '对比解码': '来源引用',
  '期望': '来源引用',
  '命中': True},
 {'样本': 'D04',
  '提示': '北京暴雨出行应优先',
  'expert基线': '注意',
  '对比解码': '查看预警',
  '期望': '查看预警',
  '命中': True},
 {'样本': 'D05',
  '提示': '代码审查发现密钥后应',
  'expert基线': '进行',
  '对比解码': '立即轮换',
  '期望': '立即轮换',
  '命中': True}]
具体答案命中：expert 基线 0/5，带 mask 对比解码 5/5


## 6. 失败案例与修正：无约束差分会奖励怪词

怪词的 amateur 概率极低，因此 `-α log p_amateur` 会给它巨大正奖励；如果直接在全词表取最大对比分数，五条都可能选择专家本来极不相信的怪词。下面真实运行无 mask 版本，再与专家可信 mask 的修正结果逐条对照。

In [6]:
failure_rows = []  # 收集无可信约束时的异常选择
for item in cases:  # 遍历所有真实生成场景
    expert_logp, amateur_logp, scores = contrastive_scores(item)  # 计算未加 mask 的全候选对比分数
    raw_index = max(range(len(scores)), key=lambda index: scores[index])  # 错误地在整个词表选择最大分
    raw_choice = item["tokens"][raw_index]  # 取得被业余模型低概率过度奖励的 token
    fixed_choice, kept_tokens, masked_scores = decode_with_mask(item)  # 使用专家可信集合修正异常选择
    failure_rows.append({"样本": item["id"], "无约束选择": raw_choice, "expert_logit": item["expert"][raw_index], "修正选择": fixed_choice, "怪词被过滤": raw_choice not in kept_tokens})  # 保存失败证据与修复结果
print("失败复现：无 plausibility mask 的选择：")  # 输出失败案例标题
pprint(failure_rows, sort_dicts=False)  # 展示专家低概率怪词为何不能参与最终竞争

失败复现：无 plausibility mask 的选择：
[{'样本': 'D01',
  '无约束选择': '量子香蕉',
  'expert_logit': -2.0,
  '修正选择': '原路退回',
  '怪词被过滤': True},
 {'样本': 'D02',
  '无约束选择': '蓝色三角',
  'expert_logit': -2.4,
  '修正选择': '执行计划',
  '怪词被过滤': True},
 {'样本': 'D03',
  '无约束选择': '月光端口',
  'expert_logit': -2.2,
  '修正选择': '来源引用',
  '怪词被过滤': True},
 {'样本': 'D04',
  '无约束选择': '递归云朵',
  'expert_logit': -2.5,
  '修正选择': '查看预警',
  '怪词被过滤': True},
 {'样本': 'D05',
  '无约束选择': '七维饼干',
  'expert_logit': -2.3,
  '修正选择': '立即轮换',
  '怪词被过滤': True}]


## 7. 生产差距与最小回归检查

真实部署必须确保两个模型使用完全相同的 tokenizer、prompt、当前位置和候选词表，否则概率无法逐 token 相减。双模型前向会增加首 token 延迟、显存和 KV cache，实际系统可评估更小 amateur、共享层或只在关键步启用。还要在事实性、重复度、开放问答和安全集上联合调 alpha 与阈值，并监控 mask 过窄导致的退化。最后的断言只验证本实验展示的高频偏好、可信集合和怪词失败分支。

In [7]:
assert len(cases) >= 5  # 确认真实生成场景数量满足逐样本教学要求
assert baseline_hits == 0  # 确认 expert 贪心基线在受控数据中真实偏向空泛 token
assert contrastive_hits == len(cases)  # 确认带可信 mask 的对比解码命中全部领域具体词
assert all(row["怪词被过滤"] for row in failure_rows)  # 确认无约束方案选出的异常词都被专家阈值排除
assert all(row["修正选择"] == item["desired"] for row, item in zip(failure_rows, cases))  # 确认过滤异常候选后恢复期望选择
assert all(len(row["保留候选"]) >= 2 for row in mask_preview)  # 确认可信集合仍保留多个候选进行真实重排
print("回归检查通过：双模型打分、可信候选重排与怪词过滤均已验证。")  # 输出最终验收结论

回归检查通过：双模型打分、可信候选重排与怪词过滤均已验证。
